A notebook for prompting of SOTA LLMs.

## Gemini

In [ ]:
# !pip install google-genai

In [ ]:
from google import genai
from google.genai import types
import json

with open("not_to_upload.txt", "r") as f:
    API_KEY = f.read().split("\n")[0]

In [ ]:

# 1. Setup
client = genai.Client(api_key=API_KEY)

# 2. Prepare the System Prompt
# Replace [TAXONOMY...] with your actual definitions
taxonomy_text = """
- Physical Artefact: Man-made objects.
- Quantity: Numbers with units or dimensionless metrics.
- Method: A procedure or technique.
- Location: A geographic place.
"""

system_instruction = f"""
### ROLE
You are an expert Named Entity Recognition (NER) system...
(Paste the full prompt from above here)
...
### TAXONOMY & RULES
{taxonomy_text}
...
"""

# 3. User Input
user_input = "The experimental results showed that the pH levels dropped by 50% in the Danube River."

# 4. Call API with JSON enforcement
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=user_input,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        response_mime_type="application/json", # Forces valid JSON output
        temperature=0.1 # Low temp is better for extraction tasks
    )
)

# 5. Process Result
try:
    data = json.loads(response.text)
    
    # Example logic to find spans (naive string match)
    for item in data:
        entity = item['entity_text']
        start_index = user_input.find(entity)
        if start_index != -1:
            end_index = start_index + len(entity)
            print(f"Found '{entity}' ({item['category']}) at [{start_index}:{end_index}]")
            print(f"Reason: {item['reasoning']}\n")
        else:
            print(f"Warning: Could not locate '{entity}' exactly in source.")
            
except json.JSONDecodeError:
    print("Model failed to generate valid JSON.")
    print(response.text)